# 08 Chat RL Walkthrough

This notebook explains the optional reinforcement-learning stage in [`scripts/chat_rl.py`](../scripts/chat_rl.py).

The real nanochat path is:

```text
SFT chat model
    -> take a GSM8K math question
    -> hide the known assistant answer
    -> sample several candidate answers from the model
    -> reward candidates that end with the correct numerical answer
    -> compare each reward with the group's average reward
    -> increase the probability of better-than-average sampled tokens
    -> decrease the probability of worse-than-average sampled tokens
    -> save an RL checkpoint
```

This stage is **not** included in [`runs/speedrun.sh`](../runs/speedrun.sh). The speedrun stops after SFT. Run chat RL separately if you want to experiment with it.

GRPO stands for **Group Relative Policy Optimization**:

- **Group**: generate several answers for the same prompt.
- **Relative**: score each answer relative to the group's average score.
- **Policy Optimization**: update the model so better-than-average answers become more likely and worse-than-average answers become less likely.

The source file calls this method `"GRPO"` in quotes. Its own module comment explains that nanochat deliberately uses a simpler, on-policy update closer to REINFORCE: no reference model, no KL penalty, and no PPO clipping.

The executable cells below use the real tokenizer completion renderer and the real GSM8K answer parser. They intentionally do not download GSM8K or load the multi-gigabyte SFT model.

In [1]:
from pathlib import Path
import json
import os
import sys

repo_root = Path.cwd()
if not ((repo_root / "nanochat").exists() and (repo_root / "pyproject.toml").exists()):
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "nanochat").exists() and (candidate / "pyproject.toml").exists():
            repo_root = candidate
            break

os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

missing = []
for package in ["torch", "rustbpe", "tiktoken", "tokenizers", "datasets"]:
    try:
        __import__(package)
    except Exception:
        missing.append(package)

if missing:
    raise RuntimeError(
        "Missing dependencies: " + ", ".join(missing) + "\n"
        "From the repo root run:\n"
        "  uv sync --extra cpu --group dev\n"
        "or, on CUDA machines:\n"
        "  uv sync --extra gpu --group dev\n"
        "Then reopen this notebook with the nanochat (.venv) kernel."
    )

import torch

from nanochat.common import get_base_dir
from nanochat.tokenizer import RustBPETokenizer
from tasks.gsm8k import extract_answer

artifact_root = Path(os.environ.get("NANOCHAT_ARTIFACTS_DIR", repo_root / "nanochat-artifacts")).expanduser()

def find_tokenizer_dir():
    candidates = [
        artifact_root / "tokenizer",
        Path(get_base_dir()) / "tokenizer",
    ]
    for candidate in candidates:
        if (candidate / "tokenizer.pkl").exists():
            return candidate
    return None

tokenizer_dir = find_tokenizer_dir()
tokenizer = RustBPETokenizer.from_directory(tokenizer_dir) if tokenizer_dir else None

print(f"repo root:      {repo_root}")
print(f"artifact root:  {artifact_root} (exists={artifact_root.exists()})")
print(f"base dir:       {get_base_dir()}")
print(f"tokenizer dir:  {tokenizer_dir}")

repo root:      /Users/eugene/Developer/nanochat
artifact root:  /Users/eugene/Developer/nanochat/nanochat-artifacts (exists=True)
base dir:       /Users/eugene/.cache/nanochat
tokenizer dir:  /Users/eugene/Developer/nanochat/nanochat-artifacts/tokenizer


/Users/eugene/Developer/nanochat/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 1. RL Starts From The SFT Chat Model

Base training teaches next-token prediction from general text. SFT teaches chat formatting and assistant behavior. RL starts **after** SFT because it needs a model that can already answer a user message in chat format.

The real code loads the SFT checkpoint, not the base checkpoint:

```python
model, tokenizer, meta = load_model("sft", device, phase="eval", ...)
engine = Engine(model, tokenizer)
```

See [`scripts/chat_rl.py`](../scripts/chat_rl.py#L67-L70) and the source mapping in [`nanochat/checkpoint_manager.py`](../nanochat/checkpoint_manager.py#L165-L172):

```text
"sft" -> chatsft_checkpoints
"rl"  -> chatrl_checkpoints
```

The next cell inspects metadata from the real copied-back SFT artifact without loading its large weight file.

In [2]:
sft_metadata_files = sorted((artifact_root / "chatsft_checkpoints").glob("*/meta_*.json"))

if not sft_metadata_files:
    print("No copied-back SFT metadata found under nanochat-artifacts/chatsft_checkpoints.")
    print("The walkthrough still works; this inspection cell is optional.")
else:
    sft_metadata_path = sft_metadata_files[-1]
    sft_meta = json.loads(sft_metadata_path.read_text())
    model_config = sft_meta["model_config"]
    print(f"SFT metadata: {sft_metadata_path.relative_to(artifact_root)}")
    print(f"layers:       {model_config['n_layer']}")
    print(f"embedding:    {model_config['n_embd']}")
    print(f"vocab size:   {model_config['vocab_size']}")
    print(f"sequence len: {model_config['sequence_len']}")

SFT metadata: chatsft_checkpoints/d24/meta_000483.json
layers:       24
embedding:    1536
vocab size:   32768
sequence len: 2048


## Step 2. Keep The Question, Hide The Known Answer

A GSM8K training example already contains both sides of a conversation:

```text
user:      What is 3 + 4?
assistant: 3 + 4 = 7.
           #### 7
```

RL cannot give the known answer to the model. It needs to let the model produce its own candidate answer first.

[`RustBPETokenizer.render_for_completion(...)`](../nanochat/tokenizer.py#L367-L385) deep-copies the conversation, removes the final assistant message, renders the remaining messages, and appends `<|assistant_start|>`.

The resulting prefix means: *answer this user message now*.

In [3]:
conversation = {
    "messages": [
        {"role": "user", "content": "What is 3 + 4?"},
        {"role": "assistant", "content": [
            {"type": "text", "text": "3 + 4 = 7.\n#### 7"},
        ]},
    ]
}

print("Known answer stored in the original GSM8K-like conversation:")
print(conversation["messages"][-1]["content"][-1]["text"])
print()

if tokenizer is None:
    completion_prefix_ids = None
    print("No trained tokenizer found. Expected completion prefix:")
    print("<|bos|><|user_start|>What is 3 + 4?<|user_end|><|assistant_start|>")
else:
    completion_prefix_ids = tokenizer.render_for_completion(conversation)
    print("Prefix passed to Engine.generate_batch(...):")
    print(tokenizer.decode(completion_prefix_ids))
    print()
    print(f"prefix token count: {len(completion_prefix_ids)}")

print()
print("The original conversation still contains the answer because render_for_completion deep-copies it:")
print(conversation["messages"][-1]["content"][-1]["text"])

Known answer stored in the original GSM8K-like conversation:
3 + 4 = 7.
#### 7

Prefix passed to Engine.generate_batch(...):
<|bos|><|user_start|>What is 3 + 4?<|user_end|><|assistant_start|>

prefix token count: 12

The original conversation still contains the answer because render_for_completion deep-copies it:
3 + 4 = 7.
#### 7


## Step 3. Sample Several Candidate Answers For The Same Question

The prefix from Step 2 is passed to [`Engine.generate_batch(...)`](../nanochat/engine.py#L282-L306). The real RL loop asks the current model for multiple completions of the **same** question.

There are two related arguments that are easy to mix up:

- `args.num_samples`: total number of rollouts wanted for one question. Default: `16`.
- `args.device_batch_size`: how many rollouts to generate in one batched forward/generation call. Default: `8`.

So nanochat generates the 16 rollouts in smaller chunks:

```python
num_sampling_steps = args.num_samples // args.device_batch_size

for sampling_step in range(num_sampling_steps):
    generated_token_sequences_batch, masks_batch = engine.generate_batch(
        tokens,
        num_samples=args.device_batch_size,
        max_tokens=args.max_new_tokens,
        temperature=args.temperature,
        top_k=args.top_k,
        seed=seed,
    )
    generated_token_sequences.extend(generated_token_sequences_batch)
```

See [`scripts/chat_rl.py`](../scripts/chat_rl.py#L99-L115).

With defaults:

```text
args.num_samples       = 16 total rollouts wanted
args.device_batch_size = 8 rollouts per generate_batch(...) call

num_sampling_steps = 16 // 8 = 2

first generate_batch call  -> 8 rollouts
second generate_batch call -> 8 rollouts
total                      -> 16 rollouts for this question
```

This set of candidate answers is often called a **group of rollouts**. Temperature defaults to `1.0`, rather than greedy `0.0`, because RL needs some variation: if every sample is identical, there is nothing useful to compare inside the group.

The toy example below uses four rollouts only to keep the table readable.

In [4]:
rollouts = [
    "3 + 4 = 7.\n#### 7",
    "I think the answer is 8.\n#### 8",
    "3 + 4 equals 6.\n#### 6",
    "Adding the values gives 7.\n#### 7",
]

for index, rollout in enumerate(rollouts):
    print(f"rollout {index}: {rollout!r}")

rollout 0: '3 + 4 = 7.\n#### 7'
rollout 1: 'I think the answer is 8.\n#### 8'
rollout 2: '3 + 4 equals 6.\n#### 6'
rollout 3: 'Adding the values gives 7.\n#### 7'


## Step 4. GSM8K Converts Each Candidate Into A Reward

Nanochat uses a deliberately simple reward for this experiment:

```text
correct final numerical answer -> reward 1.0
wrong final numerical answer   -> reward 0.0
```

The direct code path is:

1. [`scripts/chat_rl.py`](../scripts/chat_rl.py#L117-L126) decodes each sampled answer and asks the task for a reward:

```python
generated_text = tokenizer.decode(generated_tokens)
reward = train_task.reward(conversation, generated_text)
```

2. `train_task` is a [`GSM8K`](../tasks/gsm8k.py) instance, so that calls [`GSM8K.reward(...)`](../tasks/gsm8k.py#L110-L117):

```python
is_correct = self.evaluate(conversation, assistant_response)
return float(is_correct)
```

3. [`GSM8K.evaluate(...)`](../tasks/gsm8k.py#L87-L108) extracts the reference number from the known answer and the predicted number from the sampled answer:

```python
ref_num = extract_answer(last_text_part)
pred_num = extract_answer(assistant_response)
is_correct = int(pred_num == ref_num)
```

So in this repo, `reward(...)` is not a separate neural reward model. It is a thin wrapper around GSM8K answer checking: correct final `####` number becomes `1.0`, otherwise `0.0`.

The `####` convention comes from the actual [OpenAI GSM8K dataset](https://huggingface.co/datasets/openai/gsm8k).

In [5]:
reference_response = conversation["messages"][-1]["content"][-1]["text"]
reference_answer = extract_answer(reference_response)

rewards_list = []
print(f"reference answer: {reference_answer!r}")
print()
print("rollout | extracted answer | reward")
print("--------+------------------+-------")
for index, rollout in enumerate(rollouts):
    predicted_answer = extract_answer(rollout)
    reward = float(predicted_answer == reference_answer)
    rewards_list.append(reward)
    print(f"{index:>7} | {predicted_answer!r:<16} | {reward:.1f}")

reference answer: '7'

rollout | extracted answer | reward
--------+------------------+-------
      0 | '7'              | 1.0
      1 | '8'              | 0.0
      2 | '6'              | 0.0
      3 | '7'              | 1.0


## Step 5. Rewards Become Relative Advantages

A raw reward says whether one rollout was correct. RL needs a signal that says whether the sampled tokens should become **more** or **less** likely.

Nanochat compares each rollout with the average reward of the group:

```python
mu = rewards.mean()
advantages = rewards - mu
```

See [`scripts/chat_rl.py`](../scripts/chat_rl.py#L142-L145).

- Positive advantage: this rollout was better than the group average. Increase the probability of its sampled tokens.
- Negative advantage: this rollout was worse than the group average. Decrease the probability of its sampled tokens.
- Zero advantage: this rollout gives no update signal.

Unlike some GRPO variants, nanochat does not divide by the group's standard deviation.

In [6]:
rewards = torch.tensor(rewards_list, dtype=torch.float32)
mu = rewards.mean()
advantages = rewards - mu

print(f"rewards:    {rewards.tolist()}")
print(f"mean:       {mu.item():.2f}")
print(f"advantages: {advantages.tolist()}")

rewards:    [1.0, 0.0, 0.0, 1.0]
mean:       0.50
advantages: [0.5, -0.5, -0.5, 0.5]


## Step 6. Train Only On Tokens The Model Actually Chose

Notebook 07 introduced the masks returned by the inference engine. They become important here.

The engine returns `mask=0` for prompt tokens and calculator output inserted by the engine. It returns `mask=1` for tokens sampled from the model. [`scripts/chat_rl.py`](../scripts/chat_rl.py#L131-L139) shifts the sequence into next-token targets and turns every `mask=0` target into `-1`:

```python
inputs = ids[:, :-1]
targets = ids[:, 1:].clone()
targets[mask_ids[:, 1:] == 0] = -1
```

GPT's cross-entropy loss ignores target `-1`. The optimizer therefore updates the model only for decisions the model made itself.

Important: at this point, `ids` is the **full rollout sequence**, not just the original user prompt. It contains:

```text
prompt tokens + sampled assistant tokens + inserted calculator output tokens
```

The model already generated those assistant tokens in Step 3. During RL training, nanochat replays that full rollout through GPT so it can ask: *what log-probability did the model assign to each generated token?* The causal attention mask still prevents each position from looking ahead, and the loss mask decides which next-token targets count for learning.

| Token | Source | `token_mask` |
|---|---|---:|
| <code>&lt;&#124;python_start&#124;&gt;</code> | generated by model | `1` |
| `3` | generated by model | `1` |
| `+` | generated by model | `1` |
| `4` | generated by model | `1` |
| <code>&lt;&#124;python_end&#124;&gt;</code> | generated by model | `1` |
| <code>&lt;&#124;output_start&#124;&gt;</code> | inserted by engine | `0` |
| `7` | inserted by calculator | `0` |
| <code>&lt;&#124;output_end&#124;&gt;</code> | inserted by engine | `0` |

This is the right behavior: the model can learn to request a useful calculation, but it is not rewarded for pretending it generated the calculator's deterministic output.

The executable table below abbreviates the question as one readable row. The real BPE tokenizer represents that question with several token ids, and each real id has its own row and mask value.

In [7]:
token_labels = [
    "<|bos|>",
    "<|user_start|>",
    "[question BPE tokens]",
    "<|user_end|>",
    "<|assistant_start|>",
    "<|python_start|>",
    "3",
    "+",
    "4",
    "<|python_end|>",
    "<|output_start|>",
    "7",
    "<|output_end|>",
    "####",
    "7",
]

# Same meaning as the masks returned by Engine.generate_batch(...).
token_masks = torch.tensor([[0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1]])
ids = torch.arange(len(token_labels)).unsqueeze(0)

inputs = ids[:, :-1]
targets = ids[:, 1:].clone()
target_masks = token_masks[:, 1:]
targets[target_masks == 0] = -1

rows = []
for input_id, target_id, target_mask in zip(inputs[0], targets[0], target_masks[0]):
    input_label = token_labels[input_id]
    next_label = token_labels[input_id + 1]
    used = "yes" if target_id.item() >= 0 else "no (-1)"
    rows.append((repr(input_label), repr(next_label), str(target_mask.item()), used))

headers = ("current token in rollout", "next-token target", "mask", "used in loss?")
current_w = max(len(headers[0]), *(len(row[0]) for row in rows))
target_w = max(len(headers[1]), *(len(row[1]) for row in rows))
mask_w = max(len(headers[2]), *(len(row[2]) for row in rows))
used_w = max(len(headers[3]), *(len(row[3]) for row in rows))

print(f"{headers[0]:<{current_w}}  -> {headers[1]:<{target_w}}  {headers[2]:>{mask_w}}  {headers[3]:<{used_w}}")
print(f"{'-' * current_w}--+--{'-' * target_w}--+-{'-' * mask_w}--+--{'-' * used_w}")
for current_token, next_token, mask_value, used in rows:
    print(f"{current_token:<{current_w}}  -> {next_token:<{target_w}}  {mask_value:>{mask_w}}  {used:<{used_w}}")

current token in rollout  -> next-token target        mask  used in loss?
--------------------------+---------------------------+-------+---------------
'<|bos|>'                 -> '<|user_start|>'            0  no (-1)      
'<|user_start|>'          -> '[question BPE tokens]'     0  no (-1)      
'[question BPE tokens]'   -> '<|user_end|>'              0  no (-1)      
'<|user_end|>'            -> '<|assistant_start|>'       0  no (-1)      
'<|assistant_start|>'     -> '<|python_start|>'          1  yes          
'<|python_start|>'        -> '3'                         1  yes          
'3'                       -> '+'                         1  yes          
'+'                       -> '4'                         1  yes          
'4'                       -> '<|python_end|>'            1  yes          
'<|python_end|>'          -> '<|output_start|>'          0  no (-1)      
'<|output_start|>'        -> '7'                         0  no (-1)      
'7'                       -> '<|o

## Step 7. Multiply Token Log-Probabilities By Advantages

This step uses the same cross-entropy machinery as base training, but it uses it differently.

In base training, GPT computes cross-entropy against real dataset tokens:

```python
loss = F.cross_entropy(logits, targets, ignore_index=-1)
```

Cross-entropy is negative log-probability:

```text
cross_entropy = -log(P(target token))
```

So base training minimizes `-logp`, which is the same as maximizing `logp`, which is the same as making the dataset target token more likely.

RL also asks GPT for per-token cross-entropy, but now the targets are tokens the model itself sampled during the rollout. Nanochat negates cross-entropy to recover actual log-probabilities:

```python
logp = -model(inputs, targets, loss_reduction="none").view_as(inputs)
pg_obj = (logp * advantages.unsqueeze(-1)).sum()
num_valid = (targets >= 0).sum().clamp(min=1)
pg_obj = pg_obj / (num_valid * num_passes * examples_per_rank)
loss = -pg_obj
loss.backward()
```

See [`scripts/chat_rl.py`](../scripts/chat_rl.py#L258-L272).

Important sign detail:

```text
model(...) returns cross_entropy = -logp
logp = -model(...) = log(P(sampled token))
```

`logp` is usually negative because probabilities are between `0` and `1`:

```text
P = 0.90 -> logp = -0.105
P = 0.01 -> logp = -4.605
```

The advantage multiplication is the heart of the RL update:

- A sampled token from a correct, positive-advantage rollout is pushed to become more likely.
- A sampled token from an incorrect, negative-advantage rollout is pushed to become less likely.
- Prompt tokens and calculator-inserted tokens contribute zero because Step 6 changed their targets to `-1`.

A compact comparison:

| Training type | Target tokens | Loss minimized by PyTorch | Equivalent objective | What gets more likely? |
|---|---|---|---|---|
| Base training | real dataset next tokens | `-logp` | maximize `logp` | dataset tokens |
| SFT training | assistant answer tokens | `-logp`, mask user tokens | maximize assistant-token `logp` | assistant-style answer tokens |
| RL training | model-sampled rollout tokens | `-(logp * advantage)` | maximize `logp * advantage` | positive-advantage sampled tokens |

`pg_obj` is an objective to maximize. PyTorch optimizers minimize losses, so nanochat uses `loss = -pg_obj`.

With one sampled token, the sign logic is:

```text
advantage > 0:
  loss = -(logp * positive)
  minimizing loss pushes logp upward toward 0
  probability goes up

advantage < 0:
  loss = -(logp * negative)
  minimizing loss pushes logp downward, more negative
  probability goes down
```

In [8]:
# Three illustrative sampled-token log-probabilities for each rollout.
# Real logp comes from GPT for every non-ignored sampled token.
toy_logp = torch.tensor([
    [-0.2, -0.3, -0.4],  # correct rollout
    [-1.5, -1.2, -1.0],  # wrong rollout
    [-0.8, -0.9, -0.7],  # wrong rollout
    [-0.3, -0.2, -0.3],  # correct rollout
])

weighted_terms = toy_logp * advantages.unsqueeze(-1)
pg_obj = weighted_terms.sum() / toy_logp.numel()
loss = -pg_obj

print("advantages:")
print(advantages)
print()
print("logp * advantage terms:")
print(weighted_terms)
print()
print(f"policy-gradient objective to maximize: {pg_obj.item():.4f}")
print(f"loss passed to optimizer to minimize:   {loss.item():.4f}")

print()
print("One-token sign intuition:")
for probability, advantage_value in [(0.90, 0.5), (0.01, 0.5), (0.90, -0.5), (0.01, -0.5)]:
    logp_value = torch.log(torch.tensor(probability))
    pg_value = logp_value * advantage_value
    loss_value = -pg_value
    direction = "increase probability" if advantage_value > 0 else "decrease probability"
    print(
        f"P={probability:>4.2f}, logp={logp_value.item():>7.3f}, "
        f"adv={advantage_value:>4.1f}, loss={loss_value.item():>7.3f} -> {direction}"
    )

advantages:
tensor([ 0.5000, -0.5000, -0.5000,  0.5000])

logp * advantage terms:
tensor([[-0.1000, -0.1500, -0.2000],
        [ 0.7500,  0.6000,  0.5000],
        [ 0.4000,  0.4500,  0.3500],
        [-0.1500, -0.1000, -0.1500]])

policy-gradient objective to maximize: 0.1833
loss passed to optimizer to minimize:   -0.1833

One-token sign intuition:
P=0.90, logp= -0.105, adv= 0.5, loss=  0.053 -> increase probability
P=0.01, logp= -4.605, adv= 0.5, loss=  2.303 -> increase probability
P=0.90, logp= -0.105, adv=-0.5, loss= -0.053 -> decrease probability
P=0.01, logp= -4.605, adv=-0.5, loss= -2.303 -> decrease probability


## Step 8. Apply One Optimizer Update

The real loop accumulates rollout gradients across the examples assigned to the current GPU rank. After those backward passes, it updates model parameters once:

```python
optimizer.step()
model.zero_grad(set_to_none=True)
```

See [`scripts/chat_rl.py`](../scripts/chat_rl.py#L295-L301).

This is still ordinary PyTorch training. The unusual part is not `optimizer.step()`. The unusual part is how the loss was constructed from sampled model behavior and rewards before the optimizer sees it.

## Step 9. Evaluate With `pass@k`

During RL, nanochat periodically samples answers for held-out GSM8K questions. It logs `pass@k`:

```text
pass@1 = fraction of questions solved by the first sampled answer
pass@2 = fraction solved by at least one of the first two answers
pass@k = fraction solved by at least one of the first k answers
```

See [`run_gsm8k_eval(...)`](../scripts/chat_rl.py#L149-L191) and the aggregation loop in [`scripts/chat_rl.py`](../scripts/chat_rl.py#L224-L240).

This is **not** the training path. It is only measurement:

```text
training get_batch(...):
  uses GSM8K train examples
  generates rollouts
  computes rewards and advantages
  runs backward + optimizer.step()

run_gsm8k_eval(...):
  uses held-out GSM8K eval/test examples
  generates samples
  checks whether any sample is correct
  logs pass@k
  does not retry failed questions
  does not call backward or optimizer.step()
```

`pass@k` is useful because RL samples multiple candidate solutions. It tells you both how good one answer is and how useful additional attempts are. If none of the first `k` samples are correct, that question simply counts as failed for `pass@k`.

In [9]:
# Each row is one validation question. Each boolean is one sampled answer.
validation_outcomes = [
    [False, True,  False, False],
    [True,  False, False, False],
    [False, False, False, True ],
    [False, False, False, False],
]

for k in range(1, 5):
    solved = sum(any(row[:k]) for row in validation_outcomes)
    pass_at_k = solved / len(validation_outcomes)
    print(f"pass@{k}: {solved}/{len(validation_outcomes)} = {pass_at_k:.2f}")

pass@1: 1/4 = 0.25
pass@2: 2/4 = 0.50
pass@3: 2/4 = 0.50
pass@4: 3/4 = 0.75


## Step 10. Save RL Checkpoints Separately

SFT checkpoints remain under `chatsft_checkpoints/`. RL writes updated model weights under `chatrl_checkpoints/`:

```text
$NANOCHAT_BASE_DIR/
├── chatsft_checkpoints/
│   └── d24/
│       ├── model_000483.pt
│       └── meta_000483.json
└── chatrl_checkpoints/
    └── d24/
        ├── model_<step>.pt
        └── meta_<step>.json
```

The save path is built in [`scripts/chat_rl.py`](../scripts/chat_rl.py#L304-L325). Nanochat deliberately passes `None` as optimizer data for RL, so the RL checkpoint contains model weights and metadata but no optimizer-state files.

The copied-back training artifacts from the speedrun are expected to contain SFT checkpoints but no RL checkpoints: [`runs/speedrun.sh`](../runs/speedrun.sh) does not launch `scripts.chat_rl`.

In [10]:
rl_checkpoint_root = artifact_root / "chatrl_checkpoints"

if not rl_checkpoint_root.exists():
    print(f"No RL checkpoint directory found: {rl_checkpoint_root}")
    print("That is expected for artifacts produced by runs/speedrun.sh.")
else:
    rl_files = sorted(path.relative_to(artifact_root) for path in rl_checkpoint_root.rglob("*") if path.is_file())
    print("RL checkpoint files:")
    for path in rl_files:
        print(f"  {path}")

No RL checkpoint directory found: /Users/eugene/Developer/nanochat/nanochat-artifacts/chatrl_checkpoints
That is expected for artifacts produced by runs/speedrun.sh.


## Step 11. Keep This Mental Model

For each GSM8K training question, nanochat does this:

```text
1. Load one question and its known answer from GSM8K.
2. Remove the known assistant answer and keep an assistant-start prompt.
3. Ask the current SFT model to sample several candidate answers.
4. Reward each candidate with 1.0 if its final #### number is correct, else 0.0.
5. Subtract the group's mean reward to obtain relative advantages.
6. Ignore prompt tokens and calculator-inserted tokens by setting their targets to -1.
7. Multiply sampled-token log-probabilities by rollout advantages.
8. Backpropagate the negative policy-gradient objective.
9. Apply optimizer.step().
10. Periodically evaluate pass@k and save model weights under chatrl_checkpoints/.
```

The most useful source files to read next are:

- [`scripts/chat_rl.py`](../scripts/chat_rl.py): the full rollout, reward, policy-gradient, evaluation, and save loop.
- [`tasks/gsm8k.py`](../tasks/gsm8k.py): GSM8K loading, calculator-part parsing, final-answer extraction, evaluation, and reward.
- [`nanochat/tokenizer.py`](../nanochat/tokenizer.py#L367-L385): conversion from a full known conversation into a completion prompt.
- [`nanochat/engine.py`](../nanochat/engine.py#L282-L306): batched generation and model-token versus forced-token masks.
- [`nanochat/checkpoint_manager.py`](../nanochat/checkpoint_manager.py#L165-L172): source names mapped to checkpoint directories.

If the distinction between generated and calculator-inserted tokens feels fuzzy, revisit Step 3 of `07_inference_engine_walkthrough.ipynb`. That inference detail is exactly why RL can construct the right loss mask.

## How nanochat's `"GRPO"` Differs From Full GRPO

The useful mental model is GRPO-like, but [`scripts/chat_rl.py`](../scripts/chat_rl.py#L1-L18) intentionally removes several pieces usually associated with full GRPO/PPO-style RL:

| Feature | Full GRPO / PPO-style RL | nanochat `chat_rl.py` |
|---|---|---|
| Grouped samples | Yes, several completions per prompt | Yes, `num_samples` completions per GSM8K question |
| Relative advantage | Usually normalize reward within the group | Uses `reward - group_mean` only |
| Standard-deviation normalization | Often uses `(reward - mean) / std` | Does **not** divide by standard deviation |
| Reference model / KL penalty | Often keeps policy close to a reference model | No reference model and no KL penalty |
| PPO ratio and clipping | Often uses probability ratios and clipping | No ratio and no clipping because samples are on-policy |
| Implementation feel | More machinery, more stability controls | Smaller teaching implementation, closer to REINFORCE |

So in this notebook, read `GRPO` as: **sample a group, score each sample relative to the group, and push generated tokens up or down based on that relative score**. That is the part nanochat actually implements.